# NPS Space Systems LLM Tutor

**Goal:** Build an LLM-powered tutoring assistant for a graduate-level space systems course at NPS. 
The tutor answers student questions by retrieving grounded context from course lecture slides, lab documents, and the course manual using a Retrieval-Augmented Generation (RAG) pipeline. 
An agentic workflow routes queries to the appropriate vectorstore (labs, lectures, or manual) or falls back to web search when course materials are insufficient. 
The system is designed to support student learning by providing accurate, course-specific answers and, in future iterations, delivering practice questions and guided feedback aligned with course learning objectives.

# Environment setting

In [1]:
# Standard library imports
import json
import logging
import operator
import os

# Disable HuggingFace tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
from typing import Annotated, List, Literal
from tqdm import tqdm
import random

# Third-party imports
import requests
from IPython.display import Image, display
from typing_extensions import NotRequired, TypedDict

# Pydantic
from pydantic import BaseModel, Field

# OpenAI
from openai import OpenAI, AsyncOpenAI

# LangChain Core
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

# LangChain potpourri
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.agents import create_agent

# LangChain OpenAI
from langchain_openai import ChatOpenAI

# LangChain Community
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.document_loaders import PyMuPDFLoader

# LangGraph
from langgraph.graph import StateGraph, END

# ChromaDB
from chromadb.utils.embedding_functions import HuggingFaceEmbeddingServer

# RAG components
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LangChain tools
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document

# LLM
from langchain_openai import ChatOpenAI

# PDF processing
from pypdf import PdfReader
from langchain_community.document_loaders import PyMuPDFLoader

# Text processing
import textwrap
from textblob import TextBlob

# Recursive parsing
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Dynamic Semantic Chunking
from langchain_experimental.text_splitter import SemanticChunker

# Supports literal conversion
import ast

# Async tools
import asyncio
import time

# For measuring semantic similarity
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# For sparse retrieval
from langchain_community.retrievers import BM25Retriever

# For Cross-Encoder retrieval
from sentence_transformers import CrossEncoder

# Ingestion tools
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

# Add the PARENT directory (oa4910) to sys.path and load helper functions
sys.path.append(str(Path.cwd().parent))
from helper_code.rag.load_dataset import setup_embedding_function, load_db_from_dir, load_vectorstore

## Load Model(s)

In [2]:
API_KEY = "sk-UtrV9i5fFenmG6hvMss71A"
BASE_URL = "http://trac-malenia.ern.nps.edu:8080/inference/v1"

# Check the models available
model_ids = []

try:
    response = requests.get(
        f"{BASE_URL}/models",
        headers={"Authorization": f"Bearer {API_KEY}"}
    )
    response.raise_for_status()  
    info = response.json()
    
    for model in info['data']:
        model_ids.append(model['id'])
    model_id = info['data'][0]['id']
    print(f"Available Models: {model_ids}")
    print(f"\nDefault Selected Model Id: {model_id}")
except Exception as e:
    print(f"Error accessing model endpoint: {e}")

Available Models: ['TRAC-MTRY/traclm-v4-7b-instruct', 'google/gemma-3-27b-it', 'casperhansen/llama-3.3-70b-instruct-awq', 'allenai/Olmo-3-7B-Instruct', 'Qwen/Qwen3-30B-A3B-Thinking-2507-FP8', 'openai/gpt-oss-120b']

Default Selected Model Id: TRAC-MTRY/traclm-v4-7b-instruct


In [3]:
model_id = model_ids[2]
print(model_id)

# Create a model instance to use as your router -- HINT: which of the available models support tool calling?
llm_tools = ChatOpenAI(
    base_url=BASE_URL,
    model=model_id,
    api_key=API_KEY,
    temperature=0,
    max_tokens = 1024,  
    name="Llama"  # Name the LLM for langchain
)

casperhansen/llama-3.3-70b-instruct-awq


In [4]:
hfe = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False},
)

# Data Ingestion

## Import the all files (lectures, labs, manual)

In [12]:
# Lecture ingestion

directory = Path('./Lectures')
doc_paths = sorted(directory.glob('*.pptx'))
doc_paths.append(directory/'SS3861_Lectures.pdf')   # Adding the lecture note
print(doc_paths)
print("\n")

# Lab ingestion
docx_paths = sorted(Path('./Labs').rglob('*.docx'))
docx_files = [str(path.absolute()) for path in docx_paths]

print(docx_files)

[PosixPath('Lectures/SS3861 - 0 - Intro.pptx'), PosixPath('Lectures/SS3861 - 1 - Bits Bytes and Binary (Number Systems).pptx'), PosixPath('Lectures/SS3861 - 2 - Encoding.pptx'), PosixPath('Lectures/SS3861 - 3 - C&DH Intro + Comm Protocols.pptx'), PosixPath('Lectures/SS3861 - 4 - Ethernet UDP.pptx'), PosixPath('Lectures/SS3861 - 5 - Basic_Circuits.pptx'), PosixPath('Lectures/SS3861 - 6 - Computer Architecture, Failure Mitigation.pptx'), PosixPath('Lectures/SS3861 - 7 - EPS.pptx'), PosixPath('Lectures/SS3861 - 8 - Spacecraft Buses, ICDs.pptx'), PosixPath('Lectures/SS3861 - 9 - Testing.pptx'), PosixPath('Lectures/SS3861 - Lab 3 - Polling vs Blocking (Reference).pptx'), PosixPath('Lectures/SS3861_Lectures.pdf')]


['/home/justinkwerner/LLM Projects/thesis_llm/thesis_llm/Labs/Lab0_ASCII_Review/SS3861-Lab0-ASCII_Review-AY25.docx', '/home/justinkwerner/LLM Projects/thesis_llm/thesis_llm/Labs/Lab1_rPi_Intro/Lab1_AY25.docx', '/home/justinkwerner/LLM Projects/thesis_llm/thesis_llm/Labs/Lab1_rPi_

## Docling ingestion

In [13]:
# Setting up Docling with custom pipeline options to manage memory usage
pipeline_options = PdfPipelineOptions()

# Using AcceleratorOptions to set threads and hardware
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=1,  # Turn off parallel processing and process one at a time to prevent memory explosion
    device="cpu"    # CPU forced
)

pipeline_options.images_scale = 1.0    
# pipeline_options.do_ocr = False       # Whether to disable OCR entirely if text extraction from graphics is not needed.

# Creating the converter with the specified options
converter = DocumentConverter(
    format_options={
        "pdf": PdfFormatOption(pipeline_options=pipeline_options)
    }
)

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# HybridChunker handles Docling documents (structure-aware, respects headings/tables)
hybrid_chunker = HybridChunker(tokenizer=tokenizer, max_tokens=512)

# SemanticChunker will be applied AFTER Docling ingestion, on LangChain Documents
semantic_chunker = SemanticChunker(
    embeddings=hfe,
    breakpoint_threshold_type="percentile",
)

def load_with_docling(file_path: Path):
    print(f"Processing: {file_path.name}...") 
    
    # Convert file and chunk using HybridChunker (correct API for Docling documents)
    conv_result = converter.convert(file_path)
    doc = conv_result.document
    
    chunk_iter = hybrid_chunker.chunk(doc)
    
    docs = []
    for chunk in chunk_iter:
        page_num = 0
        if chunk.meta.doc_items and chunk.meta.doc_items[0].prov:
            page_num = chunk.meta.doc_items[0].prov[0].page_no
            
        metadata = {
            "source": str(file_path.resolve()),
            "filename": file_path.name,
            "page": page_num,
            "headings": chunk.meta.headings,
        }
        
        docs.append(Document(page_content=chunk.text, metadata=metadata))
        
    return docs

### Lecture ingestion with Docling

In [8]:
# Extract text into documents (List of Lists: 1 list per file)
print("Starting ingestion with Docling...")
docs_multilevel_doc = [load_with_docling(Path(doc_path)) for doc_path in doc_paths]

# Flatten docs into a 1-dimensional list (All chunks combined)
docs_flat_doc = [item for sublist in docs_multilevel_doc for item in sublist]

# Verify the extraction results
print(f'Total Chunks (Flat): {len(docs_flat_doc)}')

Starting ingestion with Docling...
Processing: SS3861_Lectures.pdf...


/home/justinkwerner/LLM Projects/thesis_llm/thesis_llm/.venv/lib/python3.12/site-packages/omegaconf/omegaconf.py:572: UserWarning: update() merge flag is is not specified, defaulting to False.
For more details, see https://github.com/omry/omegaconf/issues/367
  warnings.warn(
[INFO] 2026-03-31 09:51:03,167 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-31 09:51:03,257 [RapidOCR] download_file.py:60: File exists and is valid: /home/justinkwerner/LLM Projects/thesis_llm/thesis_llm/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-31 09:51:03,260 [RapidOCR] main.py:53: Using /home/justinkwerner/LLM Projects/thesis_llm/thesis_llm/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-31 09:51:03,744 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-31 09:51:03,756 [RapidOCR] download_file.py:60: File exists and is valid: /home/justinkwerner/LLM Projects/thesis_llm/thesi

Total Chunks (Flat): 117


In [9]:
# Convert to JSON-serializable format and save

from pathlib import Path

serializable_docs_ = []

for doc in docs_flat_doc:
    source_path = doc.metadata.get("source")
    path_obj = Path(source_path)

    serializable_docs_.append({
        "text": doc.page_content,
        "source": str(path_obj.resolve()),
        "filename": path_obj.name,
        "page": doc.metadata.get("page", 0),
        "filetype": path_obj.suffix.lower()
    })

with open("lectures.json", "w", encoding="utf-8") as f:
    json.dump(serializable_docs_, f, ensure_ascii=False, indent=2)

### Lab ingestion with docling

In [10]:
# Extract text into documents (List of Lists: 1 list per file)
print("Starting ingestion with Docling...")
docs_multilevel_lab = [load_with_docling(Path(docx_file)) for docx_file in docx_files]

# Flatten docs into a 1-dimensional list (All chunks combined)
docs_flat_lab = [item for sublist in docs_multilevel_lab for item in sublist]

# Verify the extraction results
print(f'Total Chunks (Flat): {len(docs_flat_lab)}')

Starting ingestion with Docling...
Total Chunks (Flat): 0


In [10]:
# Convert to JSON-serializable format and save

serializable_docs2_ = []

for doc in docs_flat_lab:
    source_path = doc.metadata.get("source")
    path_obj = Path(source_path)

    serializable_docs2_.append({
        "text": doc.page_content,
        "source": str(path_obj.resolve()),
        "filename": path_obj.name,
        "page": doc.metadata.get("page", 0),
        "filetype": path_obj.suffix.lower()
    })

with open("labs.json", "w", encoding="utf-8") as f:
    json.dump(serializable_docs2_, f, ensure_ascii=False, indent=2)

### Manual ingestion

In [12]:
manual_flat_docling = load_with_docling(Path('manual.pdf'))

Processing: manual.pdf...


/home/justinkwerner/LLM Project/nps_tutor_llm/.venv/lib/python3.12/site-packages/omegaconf/omegaconf.py:572: UserWarning: update() merge flag is is not specified, defaulting to False.
For more details, see https://github.com/omry/omegaconf/issues/367
  warnings.warn(
[INFO] 2026-03-11 15:28:49,915 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-11 15:28:49,999 [RapidOCR] download_file.py:60: File exists and is valid: /home/justinkwerner/LLM Project/nps_tutor_llm/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-11 15:28:50,001 [RapidOCR] main.py:53: Using /home/justinkwerner/LLM Project/nps_tutor_llm/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-11 15:28:50,488 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-11 15:28:50,504 [RapidOCR] download_file.py:60: File exists and is valid: /home/justinkwerner/LLM Project/nps_tutor_llm/.venv/lib/python3.12/site-pack

In [13]:
# Convert to JSON-serializable format and save
serializable_docs3_ = []

for doc in manual_flat_docling:
    source_path = doc.metadata.get("source")
    path_obj = Path(source_path)

    serializable_docs3_.append({
        "text": doc.page_content,
        "source": str(path_obj.resolve()),
        "filename": path_obj.name,
        "page": doc.metadata.get("page", 0),
        "filetype": path_obj.suffix.lower()
    })

with open("manual.json", "w", encoding="utf-8") as f:
    json.dump(serializable_docs3_, f, ensure_ascii=False, indent=2)

In [15]:
# Apply SemanticChunker as a second pass over the Docling/HybridChunker output.
# SemanticChunker operates on LangChain Documents, so this must come AFTER
# load_with_docling has already produced docs via HybridChunker.
print("Applying semantic chunking to lectures...")
semantic_lecture_docs = semantic_chunker.transform_documents(docs_flat_doc)
print(f"Lecture chunks after semantic chunking: {len(semantic_lecture_docs)}")

Applying semantic chunking to lectures...
Lecture chunks after semantic chunking: 598


In [16]:
print("Applying semantic chunking to labs...")
semantic_lab_docs = semantic_chunker.transform_documents(docs_flat_lab)
print(f"Lab chunks after semantic chunking: {len(semantic_lab_docs)}")

Applying semantic chunking to labs...
Lab chunks after semantic chunking: 566


In [17]:
print("Applying semantic chunking to manual...")
semantic_manual_docs = semantic_chunker.transform_documents(manual_flat_docling)
print(f"Manual chunks after semantic chunking: {len(semantic_manual_docs)}")

Applying semantic chunking to manual...
Manual chunks after semantic chunking: 613


In [18]:
def flatten_metadata(docs):
    for doc in docs:
        for key, value in doc.metadata.items():
            if isinstance(value, list):
                doc.metadata[key] = ", ".join(str(v) for v in value)
    return docs

semantic_lecture_docs = flatten_metadata(semantic_lecture_docs)
semantic_lab_docs = flatten_metadata(semantic_lab_docs)
semantic_manual_docs = flatten_metadata(semantic_manual_docs)
# Build vectorstores from semantically-chunked documents
print("\nBuilding vectorstores...")
lecture_vectorstore = Chroma.from_documents(
    documents=semantic_lecture_docs,
    embedding=hfe,
    persist_directory="./databases/chroma_lectures"
)

lab_vectorstore = Chroma.from_documents(
    documents=semantic_lab_docs,
    embedding=hfe,
    persist_directory="./databases/chroma_lab_notes"
)

manual_vectorstore = Chroma.from_documents(
    documents=semantic_manual_docs,
    embedding=hfe,
    persist_directory="./databases/chroma_manual"
)

print("\nAll vectorstores built successfully.")


Building vectorstores...

All vectorstores built successfully.


In [7]:

missing_docs = [
    Document(
        page_content=(
            "Spacecraft Communication Mission Segments: "
            "The on-orbit communications path typically includes three segments: "
            "(1) Space segment - the satellite/spacecraft in orbit, "
            "(2) Ground/User segment - ground stations and end users, "
            "(3) Subject/Target segment - the subject being observed or targeted. "
            "The Launch segment is NOT part of the on-orbit communications path — "
            "it is a separate mission phase used to deliver the spacecraft to orbit."
        ),
        metadata={"source": "course_notes", "topic": "communications_architecture"}
    )
]

lecture_vectorstore.add_documents(missing_docs)
print("Added missing content. Collection now has:", lecture_vectorstore._collection.count(), "docs")

NameError: name 'lecture_vectorstore' is not defined

## RAG Pipeline

In [74]:
def load_json_docs(json_path):
    """Load a single JSON file into LangChain Documents."""
    with open(json_path, 'r') as f:
        chunks = json.load(f)
    documents = []
    for chunk in chunks:
        if len(chunk["text"].strip()) < 20:
            continue
        doc = Document(
            page_content=chunk["text"],
            metadata={
                "source": chunk.get("filename", "unknown"),
                "page": chunk.get("page", 0),
                "filetype": chunk.get("filetype", "unknown")
            }
        )
        documents.append(doc)
    return documents

lab_docs     = load_json_docs("labs.json")
lecture_docs = load_json_docs("lectures.json")
manual_docs  = load_json_docs("manual.json")
print(f"Lab notes: {len(lab_docs)} chunks")
print(f"Lectures:  {len(lecture_docs)} chunks")
print(f"Manual:    {len(manual_docs)} chunks")


Lab notes: 292 chunks
Lectures:  343 chunks
Manual:    291 chunks


In [75]:
lab_vectorstore = Chroma(
    collection_name="langchain",
    embedding_function=hfe,
    persist_directory="./databases/chroma_lab_notes",
)

lecture_vectorstore = Chroma(
    collection_name="langchain",
    embedding_function=hfe,
    persist_directory="./databases/chroma_lectures",
)

manual_vectorstore = Chroma(
    collection_name="langchain",
    embedding_function=hfe,
    persist_directory="./databases/chroma_manual",
)

In [20]:
# Verify vectorstore chunk counts after semantic chunking
print(f"Lab notes store:  {lab_vectorstore._collection.count()} chunks")
print(f"Lectures store:   {lecture_vectorstore._collection.count()} chunks")
print(f"Manual store:     {manual_vectorstore._collection.count()} chunks")


Lab notes store:  566 chunks
Lectures store:   598 chunks
Manual store:     613 chunks


In [76]:
# Three retrievers — one per vectorstore
lab_retriever = RunnableLambda(
    lambda q: lab_vectorstore.similarity_search_with_relevance_scores(q, k=10)
)
lecture_retriever = RunnableLambda(
    lambda q: lecture_vectorstore.similarity_search_with_relevance_scores(q, k=10)
)
manual_retriever = RunnableLambda(
    lambda q: manual_vectorstore.similarity_search_with_relevance_scores(q, k=3)
)

# Combined retriever — pulls from all three stores
def retrieve_all(query):
    results = (
        lab_retriever.invoke(query) +
        lecture_retriever.invoke(query) +
        manual_retriever.invoke(query)
    )
    return results

combined_retriever = RunnableLambda(retrieve_all)


In [77]:
REWRITE_SYSTEM = (
    "You are a helpful AI assistant that specializes in rewriting complex user "
    "queries. Rewrite the following query by expanding it into multiple simpler, "
    "more concise queries. Start each new query with '[START]'. Do not provide "
    "any additional responses besides the rewritten queries."
)

rewrite_prompt = ChatPromptTemplate([
    ("system", REWRITE_SYSTEM),
    ("human", "{prompt}")
])

rewrite_chain = rewrite_prompt | llm_tools | StrOutputParser()


def rewrite_query(original: str) -> list[str]:
    """Expand one query into multiple sub-queries via the LLM."""
    raw = rewrite_chain.invoke({"prompt": original})
    sub_queries = [q.strip() for q in raw.split("[START]") if q.strip()]
    return sub_queries if sub_queries else [original]  # fallback to original


def retrieve_for_all_queries(original: str) -> str:
    """Rewrite -> retrieve for each sub-query -> deduplicate -> join context."""
    sub_queries = rewrite_query(original)

    seen, docs = set(), []
    for q in sub_queries:
        for item in combined_retriever.invoke(q):
            # Deduplicate by page_content so we don't bloat the context
            doc = item[0] if isinstance(item, tuple) else item
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                docs.append(doc)

    return "\n\n".join(doc.page_content for doc in docs)


In [78]:
# Prompt template
prompt_template = ChatPromptTemplate([
    (
        "system",
        "You are a helpful tutor for an NPS course. Use the provided context to answer "
        "student questions accurately and concisely. "
        "If the answer is not in the context, say so rather than guessing.\n\nContext: {context}\n"
    ),
    ("human", "{prompt}")
])

# Output parser
parser = StrOutputParser()
generation_chain = prompt_template | llm_tools | parser

# Full RAG chain
rag_chain = (
    {
        "context": RunnableLambda(retrieve_for_all_queries),
        "prompt": RunnableLambda(lambda x: x),
    }
    | generation_chain
)


In [79]:
# Quick RAG chain test
response = rag_chain.invoke("What is the purpose of parity bits in RS-232?")
print(response)


/tmp/ipykernel_1833/1327208114.py:9: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='fd6c68c5-f15c-4827-b520-295d90a319aa', metadata={'page': 163, 'headings': 'Video Trigger', 'filename': 'manual.pdf', 'source': '/home/justinkwerner/LLM Project/nps_tutor_llm/manual.pdf'}, page_content='Standard, Description = Triggers on a NTSC signal.. , Side = 625/PAL. , Description = Triggers on a PAL signal.. , Side = SECAM. , Description = Triggers on a SECAM signal.'), 0.09753684399406126), (Document(id='be96a70d-1261-473b-96d6-647dff96911e', metadata={'headings': 'Vertical', 'filename': 'manual.pdf', 'source': '/home/justinkwerner/LLM Project/nps_tutor_llm/manual.pdf', 'page': 188}, page_content='Pattern with pulse width quali fi cation, minimum rearm time: the time that a logic pattern must be invalid before a new occurrence of the pattern will be recognized.. Rearm Time, typical, Sensitivity.±0.2 divisions.±20 mV.±200 mV.N/A = State minimum rearm time: the time betwee

The purpose of parity bits in RS-232 is to provide a simple, low-level error checking mechanism. The parity bit is an additional bit added to the data packet to ensure that the total number of 1s in the data and parity bit is either even (for even parity) or odd (for odd parity). This allows the receiver to detect errors that may have occurred during transmission, such as a single bit flip. If the parity bit is set to "even" and the sum of the data and parity bit is not even, or if the parity bit is set to "odd" and the sum is not odd, the receiver can detect that an error has occurred.


## Agent Setup

In [80]:
# Setup logging for tool creation and testing
logger = logging.getLogger("agentic_workflow")
logger.setLevel(logging.DEBUG)

# Add console handler if not already present
if not logger.handlers:
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

logger.info("Logger configured for tool creation and testing")


INFO - Logger configured for tool creation and testing


In [81]:
# Logging runnable to track chain execution
def logging_helper(state: dict) -> dict:
    """Log internal chain steps"""
    logger.debug(f"Intermediary State: '{state}'")
    return state

r_logger = RunnableLambda(logging_helper, name="log chain state")


### Retriever Tools

In [82]:
# Semantic retriever — Labs
def semantic_retrieve_lab_w_scores(state: dict) -> dict:
    """Retrieve lab documents with relevance scores."""
    logger.info("NODE: Semantic Retriever - Labs")
    query = state["question"]
    k = state.get("k", 3)
    logger.debug(f"  Query: '{query}'")
    logger.debug(f"  Retrieving top {k} documents")
    lab_docs_with_scores = lab_vectorstore.similarity_search_with_relevance_scores(query, k=k)
    logger.info(f"  Retrieved {len(lab_docs_with_scores)} documents")
    for i, (doc, score) in enumerate(lab_docs_with_scores):
        logger.debug(f"  Doc {i}: score={score:.3f}, source={doc.metadata.get('source', 'unknown')}, content= {doc.page_content[:50].strip()}")
    return {"documents": lab_docs_with_scores}

semantic_retriever_lab = RunnableLambda(
    semantic_retrieve_lab_w_scores,
    name="semantic_retriever_lab"
)


In [83]:
@tool(response_format="content_and_artifact")
def retrieve_context_labs(query: str):
    """Retrieve information from the lab documents to help answer a query.

    Use this tool when you need to find information about course labs.
    """
    results = semantic_retriever_lab.invoke({"question": query})
    docs_with_scores = results.get("documents", [])
    if not docs_with_scores:
        return "No relevant documents found for the query.", []
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Relevance: {score:.2f}\n"
        f"Content: {doc.page_content}"
        for doc, score in docs_with_scores
    )
    return serialized, [doc for doc, _ in docs_with_scores]


In [84]:
# Semantic retriever — Lectures
def semantic_retrieve_lecture_w_scores(state: dict) -> dict:
    """Retrieve lecture documents with relevance scores."""
    logger.info("NODE: Semantic Retriever - Lectures")
    query = state["question"]
    k = state.get("k", 3)
    logger.debug(f"  Query: '{query}'")
    logger.debug(f"  Retrieving top {k} documents")
    lecture_docs_with_scores = lecture_vectorstore.similarity_search_with_relevance_scores(query, k=k)
    logger.info(f"  Retrieved {len(lecture_docs_with_scores)} documents")
    for i, (doc, score) in enumerate(lecture_docs_with_scores):
        logger.debug(f"  Doc {i}: score={score:.3f}, source={doc.metadata.get('source', 'unknown')}, content= {doc.page_content[:50].strip()}")
    return {"documents": lecture_docs_with_scores}

semantic_retriever_lecture = RunnableLambda(
    semantic_retrieve_lecture_w_scores,
    name="semantic_retriever_lecture"
)


In [85]:
@tool(response_format="content_and_artifact")
def retrieve_context_lectures(query: str):
    """Retrieve information from the lecture slides to help answer a query.

    Use this tool for conceptual questions about electronics, circuits, 
    active/passive components, spacecraft payloads, solar cells, signal theory,
    or any course topic not specific to labs or oscilloscope operation.
    """
    results = semantic_retriever_lecture.invoke({"question": query})
    docs_with_scores = results.get("documents", [])
    if not docs_with_scores:
        return "No relevant documents found for the query.", []
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Relevance: {score:.2f}\n"
        f"Content: {doc.page_content}"
        for doc, score in docs_with_scores
    )
    return serialized, [doc for doc, _ in docs_with_scores]


In [86]:
# Semantic retriever — Manual
def semantic_retrieve_manual_w_scores(state: dict) -> dict:
    """Retrieve information from the oscilloscope user manual.

    Use this tool ONLY for questions about how to operate or use the oscilloscope
    (buttons, display settings, probes, measurements). Do NOT use for general
    electronics or circuit theory questions."""
    logger.info("NODE: Semantic Retriever - Manual")
    query = state["question"]
    k = state.get("k", 3)
    logger.debug(f"  Query: '{query}'")
    logger.debug(f"  Retrieving top {k} documents")
    manual_docs_with_scores = manual_vectorstore.similarity_search_with_relevance_scores(query, k=k)
    logger.info(f"  Retrieved {len(manual_docs_with_scores)} documents")
    for i, (doc, score) in enumerate(manual_docs_with_scores):
        logger.debug(f"  Doc {i}: score={score:.3f}, source={doc.metadata.get('source', 'unknown')}, content= {doc.page_content[:50].strip()}")
    return {"documents": manual_docs_with_scores}

semantic_retriever_manual = RunnableLambda(
    semantic_retrieve_manual_w_scores,
    name="semantic_retriever_manual"
)


In [87]:
@tool(response_format="content_and_artifact")
def retrieve_context_manuals(query: str):
    """Retrieve information from the oscilloscope manual to help answer a query.

    Use this tool when you need to find information about oscilloscopes.
    """
    results = semantic_retriever_manual.invoke({"question": query})
    docs_with_scores = results.get("documents", [])
    if not docs_with_scores:
        return "No relevant documents found for the query.", []
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Relevance: {score:.2f}\n"
        f"Content: {doc.page_content}"
        for doc, score in docs_with_scores
    )
    return serialized, [doc for doc, _ in docs_with_scores]


In [88]:
@tool
def python_calculator(code: str) -> str:
    """Execute a Python expression to perform calculations.
    
    Use this for any numeric computation, unit conversion, or binary/hex math.
    Example: 'bin(31)', '(128/255)*5', 'int(\"00011111\", 2)'
    """
    try:
        result = eval(code, {"__builtins__": {}}, {"bin": bin, "hex": hex, "int": int, "round": round})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

### Web Search Tool

In [89]:
# DuckDuckGo web search object
websearch = DuckDuckGoSearchResults(output_format="json", num_results=3)

@tool
def web_search(query: str) -> str:
    """Search the web for additional information about the query.

    Use this tool when there are no relevant documents found for the query.
    """
    results = websearch.invoke(query)
    try:
        results_list = json.loads(results)
        formatted = "\n\n".join(
            f"Title: {r['title']}\nSnippet: {r['snippet']}\nLink: {r['link']}"
            for r in results_list
        )
        return formatted
    except:
        return results


### Create Agent

In [90]:
# Bundle all tools
rag_tools = [retrieve_context_labs, retrieve_context_lectures, retrieve_context_manuals, web_search, python_calculator]


In [91]:
rag_system_prompt = '''You are a teaching assistant and tutor for students in the Spacecraft Payload Design Master's course at Naval Postgraduate School.
Use the tools provided to assist students. Try not to directly provide an answer to their questions; help them learn and build a better understanding of the material.
Provide a citation for where you found the correct information. Give document name and page or slide numbers as needed.
You may need to use more than one tool. Use retrieve_context_labs to assist with lab assignments.
Use retrieve_context_lectures to answer conceptual questions about the course.
Use retrieve_context_manuals to help with oscilloscope use. Use web_search if no relevant information is provided by the other tools.
'''


In [92]:
# Create the agent
rag_agent = create_agent(
    model=llm_tools,
    tools=rag_tools,
    system_prompt=rag_system_prompt
)

print("RAG Agent created successfully!")


RAG Agent created successfully!


### Test Agent

In [93]:
# # Test 1: Conceptual question (should use retrieve_context_lectures)
# question = "What is the Solar cell IV Curve?"

# agent_result = rag_agent.invoke({"messages": [("human", question)]})

# print("=" * 60)
# print(f"Question: {question}")
# print("=" * 60)
# for message in agent_result["messages"]:
#     message.pretty_print()


In [94]:
# # Test 2: Lab-specific question (should use retrieve_context_labs)
# question = "I'm working on lab 6. What is the Solar cell IV Curve?"

# agent_result = rag_agent.invoke({"messages": [("human", question)]})

# print("=" * 60)
# print(f"Question: {question}")
# print("=" * 60)
# for message in agent_result["messages"]:
#     message.pretty_print()


In [95]:
from tutor_evaluations import evaluate_batch, display_results
import json

In [96]:
with open('eval_questions.json', 'r', encoding='utf-8') as file:
    evaluation_questions = json.load(file)

In [97]:
results = evaluate_batch(rag_agent, evaluation_questions)

INFO - NODE: Semantic Retriever - Lectures
DEBUG -   Query: 'impact of mismatched data rate settings on spacecraft and ground station communications'
DEBUG -   Retrieving top 3 documents
INFO -   Retrieved 3 documents
DEBUG -   Doc 0: score=0.297, source=/home/justinkwerner/LLM Project/nps_tutor_llm/Lectures/SS3861_Lectures.pdf, content= . The majority of data protocols on spacecraft use
DEBUG -   Doc 1: score=0.239, source=/home/justinkwerner/LLM Project/nps_tutor_llm/Lectures/SS3861_Lectures.pdf, content= Spacecraft systems rely on a variety of communicat
DEBUG -   Doc 2: score=0.235, source=/home/justinkwerner/LLM Project/nps_tutor_llm/Lectures/SS3861_Lectures.pdf, content= summarizes several common onboard communication pr
INFO - NODE: Semantic Retriever - Lectures
DEBUG -   Query: 'UART and I2C synchronous communication protocols'
DEBUG -   Retrieving top 3 documents
INFO -   Retrieved 3 documents
DEBUG -   Doc 0: score=0.346, source=/home/justinkwerner/LLM Project/nps_tutor_llm/L

In [98]:
display_results(results)

[Q1] Binary Encoding
----------------------------------------------------------------------
Question:        Convert the decimal number 31 to binary using 1 byte.
Expected Answer: 00011111
LLM Response:    The binary representation of the decimal number 31 using 1 byte is 11111.
Result:          ❌ INCORRECT  (correctness score: 0.00)

[Q2] Hex Conversion
----------------------------------------------------------------------
Question:        Convert the decimal number 31 to hexadecimal.
Expected Answer: 0x1F
LLM Response:    The decimal number 31 is equivalent to 1f in hexadecimal.
Result:          ✅ CORRECT  (correctness score: 1.00)

[Q3] ADC Quantization
----------------------------------------------------------------------
Question:        If an ADC measures voltages from 0–5V and stores the result in one unsigned byte, what voltage corresponds to a value of 128?
Expected Answer: approximately 2.5 V
LLM Response:    The voltage corresponding to a value of 128 is approximately 2.51V.